# 2 · Où part le temps dans le PSO ?

Le PSO du cours tourne en environ 3 s. Pour savoir si ce temps est passé à calculer sur le GPU ou à autre
chose, nous avons ajouté des chronomètres dans sa boucle principale (des `cudaEvent` pour ce qui passe par
le GPU, une horloge CPU pour la boucle CPU), sans rien changer d'autre, et mesuré séparément les cinq
morceaux d'une itération :

| # | Morceau | Qui travaille |
|---|---|---|
| 1 | `kernelUpdateParticle` : déplacer les particules | GPU |
| 2 | `kernelUpdatePBest` : mettre à jour les meilleurs personnels | GPU |
| 3 | copier tous les pBest du GPU vers le CPU | bus PCIe |
| 4 | boucle qui cherche le meilleur global gBest | CPU |
| 5 | renvoyer gBest du CPU vers le GPU | bus PCIe |

Le résultat montre que le calcul GPU utile ne représente qu'environ un tiers du temps : le reste part dans
les copies entre CPU et GPU et dans la recherche du meilleur sur le CPU. C'est ce qui a guidé la conception
de notre DE : aucune copie dans la boucle, et le meilleur individu cherché sur le GPU.

## 1. Récupérer le projet

La cellule suivante place le projet dans `/content/DE_CUDA` : depuis `DE_CUDA.zip` s'il a été importé dans Colab, sinon depuis le dépôt GitHub public. Elle affiche aussi le GPU.

In [1]:
# Récupérer le projet.
# Si DE_CUDA.zip a été déposé dans Colab (icône dossier à gauche → Importer), on utilise ce dossier ;
# sinon on copie le dépôt public GitHub (aucun compte nécessaire).
import os
if os.path.exists("/content/DE_CUDA.zip") and not os.path.exists("/content/DE_CUDA/src"):
    !unzip -q -o /content/DE_CUDA.zip -d /content
if not os.path.exists("/content/DE_CUDA/src"):
    !git clone -q https://github.com/amenacer/DE_CUDA.git /content/DE_CUDA
%cd /content/DE_CUDA
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv

/content/DE_CUDA
name, driver_version, memory.total [MiB]
Tesla T4, 580.82.07, 15360 MiB


## 2. Préparer le dossier de travail

On copie les fichiers du PSO dans `src/pso_timing/`. `src/pso_original/` n'est jamais modifié : c'est la référence.

In [2]:
!mkdir -p /content/DE_CUDA/src/pso_timing
!cp /content/DE_CUDA/src/pso_original/kernel.h /content/DE_CUDA/src/pso_original/kernel.cpp /content/DE_CUDA/src/pso_original/main.cpp /content/DE_CUDA/src/pso_timing/
!ls /content/DE_CUDA/src/pso_timing

kernel.cpp  kernel.cu  kernel.h  main.cpp


## 3. Le `kernel.cu` chronométré

Même `kernel.cu` que le cours, avec un seul changement : des chronomètres dans la boucle.

**Mesurer un morceau GPU (morceaux 1, 2, 3 et 5) :**

```cpp
cudaEventRecord(e0);                  // on pose un « top départ » dans la file du GPU
kernelUpdateParticle<<<...>>>(...);   // le travail à mesurer
cudaEventRecord(e1);                  // on pose un « top arrivée »
cudaEventSynchronize(e1);             // le CPU attend que le GPU ait atteint le top arrivée
cudaEventElapsedTime(&ms, e0, e1);    // temps entre les deux tops, en ms
t_k1 += ms;                           // on additionne sur toutes les itérations
```

**Pourquoi pas une horloge CPU ?** Un lancement de kernel est **asynchrone** : le CPU lance le kernel et
continue sans attendre. Une horloge CPU ne mesurerait que le lancement (quelques microsecondes). Les
`cudaEvent` sont des repères posés dans la file du GPU : ils mesurent le vrai temps de calcul.

**Le morceau 4** (boucle CPU) est mesuré avec une horloge CPU, `std::chrono`.

In [3]:
%%writefile /content/DE_CUDA/src/pso_timing/kernel.cu
#include <cuda_runtime.h>
#include <cuda.h>
#include <math_functions.h>

#include "kernel.h"
#include <chrono>


__device__ float tempParticle1[NUM_OF_DIMENSIONS];
__device__ float tempParticle2[NUM_OF_DIMENSIONS];

/* Objective function
0: Levy 3-dimensional
1: Shifted Rastigrin's Function
2: Shifted Rosenbrock's Function
3: Shifted Griewank's Function
4: Shifted Sphere's Function
*/
/**
 * Runs on the GPU, called from the GPU.
*/
__device__ float fitness_function(float x[]) {
    float res = 0;
    float somme = 0;
    float produit = 0;

    switch (SELECTED_OBJ_FUNC)  {
        case 0:
            float y1 = 1 + (x[0] - 1)/4;
            float yn = 1 + (x[NUM_OF_DIMENSIONS-1] - 1)/4;

            res += pow(sin(phi*y1), 2);

            for (int i = 0; i < NUM_OF_DIMENSIONS-1; i++) {
                float y = 1 + (x[i] - 1)/4;
                float yp = 1 + (x[i+1] - 1)/4;
                res += pow(y - 1, 2)*(1 + 10*pow(sin(phi*yp), 2)) + pow(yn - 1, 2);
            }
            break;
        case 1:
            for (int i = 0; i < NUM_OF_DIMENSIONS; i++) {
                float zi = x[i] - 0;
                res += pow(zi, 2) - 10*cos(2*phi*zi) + 10;
            }
            res -= 330;
            break;

        case 2:
            for (int i = 0; i < NUM_OF_DIMENSIONS-1; i++) {
                float zi = x[i] - 0 + 1;
                float zip1 = x[i+1] - 0 + 1;
                res += 100 * ( pow(pow(zi, 2) - zip1, 2)) + pow(zi - 1, 2);
            }
            res += 390;
            break;
        case 3:
            for (int i = 0; i < NUM_OF_DIMENSIONS; i++) {
                float zi = x[i] - 0;
                somme += pow(zi, 2)/4000;
                produit *= cos(zi/pow(i+1, 0.5));
            }
            res = somme - produit + 1 - 180;
            break;
        case 4:
            for(int i = 0; i < NUM_OF_DIMENSIONS; i++) {
                float zi = x[i] - 0;
                res += pow(zi, 2);
            }
            res -= 450;
            break;
    }

    return res;
}

/**
 *
 * Runs on the GPU, called from the CPU or the GPU
*/
__global__ void kernelUpdateParticle(float *positions, float *velocities,
                                     float *pBests, float *gBest, float r1,
                                     float r2)
{

    int i = blockIdx.x * blockDim.x + threadIdx.x;

    // avoid an out of bound for the array
    if(i >= NUM_OF_PARTICLES * NUM_OF_DIMENSIONS)
        return;

    //float rp = getRandomClamped();
    //float rg = getRandomClamped();

    float rp = r1; // random weight for personnal =>  computed from @getRandomClamped
    float rg = r2; // random weight for global =>  computed from @getRandomClamped


    // Mise à jour de velocities et positions
    velocities[i] = OMEGA * velocities[i] +
                    c1 * rp * (pBests[i] - positions[i]) +
                    c2 * rg * (gBest[i % NUM_OF_DIMENSIONS] - positions[i]);

    // Update posisi particle
    //Mise à jour de la position de la particule courante
    //incrémentant la position de la particule courante avec la vitesse de la particule courante
    positions[i] += velocities[i];
}

/**
 * Runs on the GPU, called from the CPU or the GPU
*/
__global__ void kernelUpdatePBest(float *positions, float *pBests, float* gBest)
{
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    if(i >= NUM_OF_PARTICLES * NUM_OF_DIMENSIONS || i % NUM_OF_DIMENSIONS != 0)
        return;

    for (int j = 0; j < NUM_OF_DIMENSIONS; j++)
    {
        tempParticle1[j] = positions[i + j];
        tempParticle2[j] = pBests[i + j];
    }

    if (fitness_function(tempParticle1) < fitness_function(tempParticle2))
    {
        for (int k = 0; k < NUM_OF_DIMENSIONS; k++)
            pBests[i + k] = positions[i + k];
    }
}


extern "C" void cuda_pso(float *positions, float *velocities, float *pBests, float *gBest)
{
    int size = NUM_OF_PARTICLES * NUM_OF_DIMENSIONS;
    float *devPos, *devVel, *devPBest, *devGBest;
    float temp[NUM_OF_DIMENSIONS];

    cudaMalloc((void**)&devPos, sizeof(float) * size);
    cudaMalloc((void**)&devVel, sizeof(float) * size);
    cudaMalloc((void**)&devPBest, sizeof(float) * size);
    cudaMalloc((void**)&devGBest, sizeof(float) * NUM_OF_DIMENSIONS);

    int threadsNum = 32;
    int blocksNum = ceil(size / threadsNum);

    cudaMemcpy(devPos, positions, sizeof(float) * size, cudaMemcpyHostToDevice);
    cudaMemcpy(devVel, velocities, sizeof(float) * size, cudaMemcpyHostToDevice);
    cudaMemcpy(devPBest, pBests, sizeof(float) * size, cudaMemcpyHostToDevice);
    cudaMemcpy(devGBest, gBest, sizeof(float) * NUM_OF_DIMENSIONS, cudaMemcpyHostToDevice);

    // ===== Chronometrage : preparation =====
    cudaEvent_t e0, e1;
    cudaEventCreate(&e0); cudaEventCreate(&e1);
    float ms;
    double t_k1 = 0, t_k2 = 0, t_d2h = 0, t_cpu = 0, t_h2d = 0;
    auto T0 = std::chrono::high_resolution_clock::now();

    for (int iter = 0; iter < MAX_ITER; iter++)
    {
        // (1) kernelUpdateParticle -> t_k1
        cudaEventRecord(e0);
        kernelUpdateParticle<<<blocksNum, threadsNum>>>(devPos, devVel, devPBest, devGBest,
                                                        getRandomClamped(), getRandomClamped());
        cudaEventRecord(e1); cudaEventSynchronize(e1);
        cudaEventElapsedTime(&ms, e0, e1); t_k1 += ms;

        // (2) kernelUpdatePBest -> t_k2   (meme principe que (1))
        cudaEventRecord(e0);
        kernelUpdatePBest<<<blocksNum, threadsNum>>>(devPos, devPBest, devGBest);
        cudaEventRecord(e1); cudaEventSynchronize(e1);
        cudaEventElapsedTime(&ms, e0, e1); t_k2 += ms;

        // (3) copie pBests GPU -> CPU -> t_d2h   (une copie se mesure comme un kernel)
        cudaEventRecord(e0);
        cudaMemcpy(pBests, devPBest, sizeof(float) * size, cudaMemcpyDeviceToHost);
        cudaEventRecord(e1); cudaEventSynchronize(e1);
        cudaEventElapsedTime(&ms, e0, e1); t_d2h += ms;

        // (4) boucle CPU du gBest -> t_cpu
        //     C'est du code CPU : on utilise une horloge CPU (std::chrono), pas cudaEvent.
        auto c0 = std::chrono::high_resolution_clock::now();
        for (int i = 0; i < size; i += NUM_OF_DIMENSIONS)
        {
            for (int k = 0; k < NUM_OF_DIMENSIONS; k++)
                temp[k] = pBests[i + k];
            if (host_fitness_function(temp) < host_fitness_function(gBest))
            {
                for (int k = 0; k < NUM_OF_DIMENSIONS; k++)
                    gBest[k] = temp[k];
            }
        }

        auto c1 = std::chrono::high_resolution_clock::now();
        t_cpu += std::chrono::duration<double, std::milli>(c1 - c0).count();

        // (5) copie gBest CPU -> GPU -> t_h2d
        cudaEventRecord(e0);
        cudaMemcpy(devGBest, gBest, sizeof(float) * NUM_OF_DIMENSIONS, cudaMemcpyHostToDevice);
        cudaEventRecord(e1); cudaEventSynchronize(e1);
        cudaEventElapsedTime(&ms, e0, e1); t_h2d += ms;
    }

    // ===== Chronometrage : resultats =====
    auto T1 = std::chrono::high_resolution_clock::now();
    double total = std::chrono::duration<double, std::milli>(T1 - T0).count();
    double somme = t_k1 + t_k2 + t_d2h + t_cpu + t_h2d;
    printf("\n%-22s %10s %8s\n", "Morceau", "ms", "%");
    printf("%-22s %10.1f %7.1f%%\n", "kernelUpdateParticle", t_k1,  100*t_k1/total);
    printf("%-22s %10.1f %7.1f%%\n", "kernelUpdatePBest",    t_k2,  100*t_k2/total);
    printf("%-22s %10.1f %7.1f%%\n", "memcpy pBests D->H",   t_d2h, 100*t_d2h/total);
    printf("%-22s %10.1f %7.1f%%\n", "boucle CPU gBest",     t_cpu, 100*t_cpu/total);
    printf("%-22s %10.1f %7.1f%%\n", "memcpy gBest H->D",    t_h2d, 100*t_h2d/total);
    printf("%-22s %10.1f %7.1f%%\n", "SOMME des parties",    somme, 100*somme/total);
    printf("%-22s %10.1f\n",         "TOTAL boucle",         total);
    cudaEventDestroy(e0); cudaEventDestroy(e1);

    cudaMemcpy(positions, devPos, sizeof(float) * size, cudaMemcpyDeviceToHost);
    cudaMemcpy(velocities, devVel, sizeof(float) * size, cudaMemcpyDeviceToHost);
    cudaMemcpy(pBests, devPBest, sizeof(float) * size, cudaMemcpyDeviceToHost);
    cudaMemcpy(gBest, devGBest, sizeof(float) * NUM_OF_DIMENSIONS, cudaMemcpyDeviceToHost);

    cudaFree(devPos); cudaFree(devVel); cudaFree(devPBest); cudaFree(devGBest);
}


Overwriting /content/DE_CUDA/src/pso_timing/kernel.cu


Contrôle : le fichier doit contenir **8** `cudaEventRecord` (2 par morceau GPU × 4 morceaux).

In [4]:
!grep -c cudaEventRecord /content/DE_CUDA/src/pso_timing/kernel.cu

8


## 4. Compiler et exécuter

Les warnings sur `math_functions.h` et `#546-D` sont normaux (ils viennent du code du cours). Le tableau des temps s'affiche à la fin.

In [5]:
%cd /content/DE_CUDA/src/pso_timing
!nvcc -O2 -arch=sm_75 -o pso_timing main.cpp kernel.cpp kernel.cu
!./pso_timing

/content/DE_CUDA/src/pso_timing
In file included from kernel.cu:3:
/usr/local/cuda/bin/../targets/x86_64-linux/include/math_functions.h:54:2: warning: #warning "math_functions.h is an internal header file and must not be used directly.  This file will be removed in a future CUDA release.  Please use cuda_runtime_api.h or cuda_runtime.h instead." []8;;https://gcc.gnu.org/onlinedocs/gcc/Warning-Options.html#index-Wcpp-Wcpp]8;;]
   54 | #warning "math_functions.h is an internal header file and must not be used directly.  This file will be removed in a future CUDA release.  Please use cuda_runtime_api.h or cuda_runtime.h instead."
      |  ^~~~~~~
kernel.cu(27): warning #546-D: transfer of control bypasses initialization of:
            variable "y1" (declared at line 29)
            variable "yn" (declared at line 30)
      switch (SELECTED_OBJ_FUNC) {
      ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

In file included from kernel.cu:3:
/usr/local/

### Lire le résultat

Pour chaque morceau, le tableau ci-dessus donne son temps total (ms) et sa part du temps de la boucle (%).

- Le calcul GPU utile (les deux kernels) ne fait qu'**environ un tiers** du temps.
- Copier 12 octets (gBest) coûte presque autant que copier 6 Ko (les pBest) : un transfert coûte surtout
  sa **latence fixe**.
- Environ un cinquième du temps n'est attribué à aucun morceau : c'est le coût de la mesure elle-même
  (4 synchronisations par itération). Le total chronométré dépasse donc les ~3 s du programme d'origine.
- **Conséquences pour notre DE** : aucune copie dans la boucle, meilleur individu trouvé sur le GPU,
  peu de kernels.